In [10]:
from google import genai
from pydantic import BaseModel, Field
from typing import Union, Literal, Optional
from dotenv import load_dotenv
from netmiko import ConnectHandler
import os
import json

load_dotenv(override=True)

# --- 1. Define Schemas ---
class DangerousConfig(BaseModel):
    hostname: str = Field(description="The hostname of the device provided in the prompt.")
    risk_level: Literal["High", "Critical"] = Field(description="Severity of the change.")
    impact_analysis: str = Field(description="Explanation of what will break.")
    confidence_score: float = Field(description="Score from 0.0 to 1.0.")

class StandardChange(BaseModel):
    hostname: str = Field(description="The hostname of the device provided in the prompt.")
    change_summary: str = Field(description="Brief summary of the change.")
    affected_interface: Optional[str] = Field(description="Interface being modified, if any.")
    confidence_score: float = Field(description="Score from 0.0 to 1.0.")

class ConfigAnalysisResult(BaseModel):
    decision: Union[DangerousConfig, StandardChange]

# --- 2. The Execution Layer (Netmiko) ---
def get_device_credentials(hostname: str):
    """Loads credentials from inventory.json"""
    try:
        with open("inventory.json", "r") as f:
            inventory = json.load(f)
        return inventory.get(hostname)
    except FileNotFoundError:
        print("❌ Error: inventory.json not found.")
        return None

def device_standard_change(hostname: str, commands: str):
    """
    Connects to the device and pushes the config using Netmiko.
    """
    print(f"\n⚡ INITIALIZING NETMIKO FOR: {hostname}")
    
    device_params = get_device_credentials(hostname)
    
    if not device_params:
        print(f"❌ Error: Device '{hostname}' not found in inventory.")
        return

    try:
        print(f"🔌 Connecting to {device_params['host']}...")
        
        # Context Manager ensures connection closes automatically
        with ConnectHandler(**device_params) as net_connect:
            print("🔓 Connection Successful. Entering Config Mode...")
            
            # Netmiko expects a list of commands
            config_set = commands.splitlines() 
            
            # Send the config
            output = net_connect.send_config_set(config_set)
            
            # Save configuration (optional, usually good practice)
            # net_connect.save_config() 
            
            print("\n📄 --- DEVICE TERMINAL OUTPUT ---")
            print(output)
            print("----------------------------------")
            print("✅ Configuration Pushed Successfully.")

    except Exception as e:
        print(f"❌ Netmiko Failed: {e}")


# --- 3. The Logic Layer (The Gatekeeper) ---
def process_config_change(result: ConfigAnalysisResult, raw_commands: str, threshold=0.9):
    """
    Decides if we block, push, or review.
    """
    decision = result.decision
    score = decision.confidence_score
    
    if score < threshold:
        print(f"⚠️  STATUS: PENDING APPROVAL (Low Confidence: {score})")
        return

    # If confidence is high...
    if isinstance(decision, DangerousConfig):
        print(f"⛔ ACTION: REJECT PUSH - Risk Level: {decision.risk_level}")
        print(f"Impact: {decision.impact_analysis}")
        
    elif isinstance(decision, StandardChange):
        print("🚀 DECISION: STANDARD CHANGE APPROVED.")
        print(f"Summary: {decision.change_summary}")
        
        # --- TRIGGER THE FUNCTION ---
        print("🔄 Triggering Automation...")
        device_standard_change(decision.hostname, raw_commands)

# --- 4. Run the Model ---
client = genai.Client()

# INPUTS
hostname_input = "D1"
config_snippet_input = """
interface Loopback 1000
description Test Gemini Agent
"""
# config_snippet_input = """
# no router bgp 100
# """
prompt = f"""
You are a Network Engineering expert. Analyze the following configuration snippet for safety.

Target Device Hostname: {hostname_input}
Config Snippet: '{config_snippet_input}'

Instructions:
1. Always include the 'hostname' provided above in your JSON response.
2. If the config causes an outage or drops connectivity, it is Dangerous.
3. If it is benign (description change, new interface), it is Standard.
"""

response = client.models.generate_content(
    model="gemini-2.0-flash-exp", 
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": ConfigAnalysisResult.model_json_schema(),
    },
)

# --- 5. Validate and Process ---
try:
    print("\n### LLM Analysis ###")
    # print(response.text)
    final_result = ConfigAnalysisResult.model_validate_json(response.text)
    print(f"Decision: {type(final_result.decision).__name__}")
    print(f"Response: {final_result.decision}")
    
    
    # Pass the result AND the original raw commands to the processor
    process_config_change(final_result, config_snippet_input, threshold=0.9) 
    
except Exception as e:
    print(f"Error: {e}")



### LLM Analysis ###
Decision: StandardChange
Response: hostname='D1' change_summary='The configuration snippet adds a new loopback interface, Loopback1000, and sets its description. This is a standard configuration change.' affected_interface='Loopback1000' confidence_score=1.0
🚀 DECISION: STANDARD CHANGE APPROVED.
Summary: The configuration snippet adds a new loopback interface, Loopback1000, and sets its description. This is a standard configuration change.
🔄 Triggering Automation...

⚡ INITIALIZING NETMIKO FOR: D1
🔌 Connecting to devnetsandboxiosxec9k.cisco.com...
🔓 Connection Successful. Entering Config Mode...

📄 --- DEVICE TERMINAL OUTPUT ---

Devnet-Device#configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
Devnet-Device(config)#
Devnet-Device(config)#interface Loopback 1000
Devnet-Device(config-if)#description Test Gemini Agent
Devnet-Device(config-if)#end
Devnet-Device#
----------------------------------
✅ Configuration Pushed Successfully.
